In [134]:
import json
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
labels_path = Path("./yes_1.xlsx")
df = pd.read_excel(labels_path)

In [136]:
df

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Страница 1,...,Unnamed: 310,Unnamed: 311,Unnamed: 312,Unnamed: 313,Unnamed: 314,Unnamed: 315,Unnamed: 316,Страница 56,Unnamed: 318,Страница 57
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.1 Площадь штриховки (Одиночный выбор),14.2 Нажим (Одиночный выбор),Шея отделена от тела линией (Одиночный выбор)
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Заштрихованное облако/4,Пейзаж/5,Пейзаж преобладает над изображением человека/6,Дождь,Солнце,Земля,Другое,NaN,NaN,NaN
2,ID ответа,Дата ответа,Затраченное время,Источник ответа,Источник ответа,Респондент,IP-hash,UA-hash,Логин,Код,...,249,250,251,252,1058,1059,1060,1062,1063,253
3,1211956,2025.07.25 14:19,00:06:21,Доп. ссылка,Доп. ссылка: Трифонова Виктория Вадимовна,NaN,127.0.0.1,127.0.0.1,NaN,САМ2_13лм7кл_49,...,NaN,NaN,NaN,NaN,NaN,Земля,NaN,NaN,NaN,Просто линия
4,1211954,2025.07.25 10:54,00:07:29,Доп. ссылка,Доп. ссылка: Коваль Оксана Владимировна,NaN,127.0.0.1,127.0.0.1,NaN,САМ2_7лм2кл_5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7001,1142510,2025.03.28 19:29,00:15:20,Доп. ссылка,Доп. ссылка: Мосенцова Анна Олеговна,NaN,127.0.0.1,127.0.0.1,NaN,ЧУВ5лждс (76),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Просто линия
7002,1142503,2025.03.28 19:12,00:31:32,Доп. ссылка,Доп. ссылка: Мосенцова Анна Олеговна,NaN,127.0.0.1,127.0.0.1,NaN,ЧУВ5лждс (75),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ничего из перечисленного
7003,1133195,2025.03.24 21:22,00:10:04,Доп. ссылка,Доп. ссылка: Клопотова Екатерина Евгеньевна.,NaN,127.0.0.1,127.0.0.1,NaN,ЧУВ6лмдс,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Просто линия
7004,1113127,2025.02.18 17:04,02:29:39,Доп. ссылка,Доп. ссылка: Web-ссылка,NaN,127.0.0.1,127.0.0.1,NaN,1,...,NaN,Пейзаж/5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Галстук


In [137]:
df.shape

(7006, 320)

In [138]:
df.columns.to_list()

['Unnamed: 0',
 'Unnamed: 1',
 'Unnamed: 2',
 'Unnamed: 3',
 'Unnamed: 4',
 'Unnamed: 5',
 'Unnamed: 6',
 'Unnamed: 7',
 'Unnamed: 8',
 'Страница 1',
 'Страница 2',
 'Страница 3',
 'Страница 5',
 'Unnamed: 13',
 'Unnamed: 14',
 'Unnamed: 15',
 'Unnamed: 16',
 'Unnamed: 17',
 'Unnamed: 18',
 'Unnamed: 19',
 'Unnamed: 20',
 'Unnamed: 21',
 'Unnamed: 22',
 'Unnamed: 23',
 'Unnamed: 24',
 'Unnamed: 25',
 'Unnamed: 26',
 'Unnamed: 27',
 'Unnamed: 28',
 'Страница 6',
 'Страница 7',
 'Страница 8',
 'Страница 9',
 'Страница 10',
 'Страница 11',
 'Страница 12',
 'Страница 13',
 'Unnamed: 37',
 'Unnamed: 38',
 'Unnamed: 39',
 'Страница 14',
 'Unnamed: 41',
 'Unnamed: 42',
 'Unnamed: 43',
 'Unnamed: 44',
 'Unnamed: 45',
 'Unnamed: 46',
 'Unnamed: 47',
 'Unnamed: 48',
 'Unnamed: 49',
 'Unnamed: 50',
 'Unnamed: 51',
 'Страница 15',
 'Страница 16',
 'Unnamed: 54',
 'Unnamed: 55',
 'Unnamed: 56',
 'Unnamed: 57',
 'Unnamed: 58',
 'Unnamed: 59',
 'Unnamed: 60',
 'Unnamed: 61',
 'Unnamed: 62',
 'Unnamed

In [139]:
# Удаляем ненужные колонки

columns_to_drop = [
    df["ID ответа"].name,
    df["Дата ответа"].name,
    df["Затраченное время"].name,
    df["Источник ответа"].name,
    df["Источник ответа.1"].name,
    df["Респондент"].name,
    df["IP-hash"].name,
    df["UA-hash"].name,
    df["Логин"].name,
]

df.drop(columns=columns_to_drop, inplace=True)
df

Exception ignored in: <function ZipFile.__del__ at 0x000002061EF84AF0>
Traceback (most recent call last):
  File "d:\Conda\CondaEnvs\diploma-env\lib\zipfile.py", line 1847, in __del__
    self.close()
  File "d:\Conda\CondaEnvs\diploma-env\lib\zipfile.py", line 1864, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file


KeyError: 'ID ответа'

In [ ]:
df.iloc[0]

Страница 1      Укажите код-идентификатор обследуемого (Свобод...
Страница 2         Укажите возраст обследуемого (Свободный ответ)
Страница 3             Укажите пол обследуемого (Одиночный выбор)
Страница 5         1 Расположение на листе  (Множественный выбор)
Unnamed: 13                                                   NaN
                                      ...                        
Unnamed: 315                                                  NaN
Unnamed: 316                                                  NaN
Страница 56              14.1 Площадь штриховки (Одиночный выбор)
Unnamed: 318                         14.2 Нажим (Одиночный выбор)
Страница 57         Шея отделена от тела линией (Одиночный выбор)
Name: 0, Length: 311, dtype: object

In [ ]:
df.iloc[0].to_list()

['Укажите код-идентификатор обследуемого (Свободный ответ)',
 'Укажите возраст обследуемого (Свободный ответ)',
 'Укажите пол обследуемого (Одиночный выбор)',
 '1 Расположение на листе\xa0 (Множественный выбор)',
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 '1.2 Степень смещения вправо (Одиночный выбор)',
 '1.3 Степень смещения влево (Одиночный выбор)',
 '1.4 Степень смещения вверх (Одиночный выбор)',
 '1.5 Степень смещения вниз (Одиночный выбор)',
 '1.6\xa0Размещение в углу (Одиночный выбор)',
 '1.7\xa0Что обрезано (Множественный выбор)',
 nan,
 nan,
 nan,
 '2.\xa0Размер рисунка (Одиночный выбор)',
 '3.\xa0Ракурс (Одиночный выбор)',
 'Поза (Одиночный выбор)',
 '4. Фигура в движении (Одиночный выбор)',
 '5.\xa0Способ изображения (Одиночный выбор)',
 '6.\xa0Изображение персонажа (Одиночный выбор)',
 '6.1\xa0Персонаж (Одиночный выбор)',
 '7.\xa0Нажим (Множественный выбор)',
 nan,
 nan,
 nan,
 '8. Особенности линии (Множественный выбор)',
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 '8.2\xa0Множест

In [ ]:
# Удаляем первую строку

df.columns = df.iloc[0]
df = df.iloc[1:].reset_index(drop=True)
df

,Укажите код-идентификатор обследуемого (Свободный ответ),Укажите возраст обследуемого (Свободный ответ),Укажите пол обследуемого (Одиночный выбор),1 Расположение на листе (Множественный выбор),NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.1 Площадь штриховки (Одиночный выбор),14.2 Нажим (Одиночный выбор),Шея отделена от тела линией (Одиночный выбор)
0,NaN,NaN,NaN,Без особенностей/1,Смещение вправо/2,Смещение влево/3,Смещение вверх/4,Смещение вниз/5,Размещение в углу/6,Изображение обрезано - есть место для дорисовк...,...,Заштрихованное облако/4,Пейзаж/5,Пейзаж преобладает над изображением человека/6,Дождь,Солнце,Земля,Другое,NaN,NaN,NaN
1,Код,Возраст,Пол,1,2,3,4,5,6,7,...,249,250,251,252,1058,1059,1060,1062,1063,253
2,ЧУВ6лм1кл_6,6,мужской,NaN,NaN,Смещение влево/3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
3,ЧУВ6лм1кл_5,6,мужской,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
4,ЧУВ6лм1кл_4,6,мужской,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8579,ЧУВ5лждс (76),5,женский,Без особенностей/1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Просто линия
8580,ЧУВ5лждс (75),5,женский,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ничего из перечисленного
8581,ЧУВ6лмдс,6,мужской,NaN,NaN,NaN,Смещение вверх/4,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Просто линия
8582,1,1,женский,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,Пейзаж/5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Галстук


In [ ]:
for col, val in df.iloc[0].items():
    print(f"col={repr(col)} | val={repr(val)}")

col='Укажите код-идентификатор обследуемого (Свободный ответ)' | val=nan
col='Укажите возраст обследуемого (Свободный ответ)' | val=nan
col='Укажите пол обследуемого (Одиночный выбор)' | val=nan
col='1 Расположение на листе\xa0 (Множественный выбор)' | val='Без особенностей/1'
col=nan | val='Смещение вправо/2'
col=nan | val='Смещение влево/3'
col=nan | val='Смещение вверх/4'
col=nan | val='Смещение вниз/5'
col=nan | val='Размещение в углу/6'
col=nan | val='Изображение обрезано - есть место для дорисовки, но не дорисовано'
col=nan | val='Изоброжение обрезано - обрезан краем листа/ нет места для дорисовки'
col='1.2 Степень смещения вправо (Одиночный выбор)' | val=nan
col='1.3 Степень смещения влево (Одиночный выбор)' | val=nan
col='1.4 Степень смещения вверх (Одиночный выбор)' | val=nan
col='1.5 Степень смещения вниз (Одиночный выбор)' | val=nan
col='1.6\xa0Размещение в углу (Одиночный выбор)' | val=nan
col='1.7\xa0Что обрезано (Множественный выбор)' | val='голова/1'
col=nan | val='одна 

In [ ]:
cols = pd.Series(df.columns, dtype="object")
vals = df.iloc[0]

filled = cols.ffill()

new_columns = []
for c, v, base in zip(cols, vals, filled):
    if pd.isna(v):
        new_columns.append("" if pd.isna(base) else str(base))
    else:
        new_columns.append(f"{base} ({v})")

df.columns = new_columns
df = df.iloc[1:].reset_index(drop=True)
df

,Укажите код-идентификатор обследуемого (Свободный ответ),Укажите возраст обследуемого (Свободный ответ),Укажите пол обследуемого (Одиночный выбор),1 Расположение на листе (Множественный выбор) (Без особенностей/1),1 Расположение на листе (Множественный выбор) (Смещение вправо/2),1 Расположение на листе (Множественный выбор) (Смещение влево/3),1 Расположение на листе (Множественный выбор) (Смещение вверх/4),1 Расположение на листе (Множественный выбор) (Смещение вниз/5),1 Расположение на листе (Множественный выбор) (Размещение в углу/6),"1 Расположение на листе (Множественный выбор) (Изображение обрезано - есть место для дорисовки, но не дорисовано)",...,14. Фон (Множественный выбор) (Заштрихованное облако/4),14. Фон (Множественный выбор) (Пейзаж/5),14. Фон (Множественный выбор) (Пейзаж преобладает над изображением человека/6),14. Фон (Множественный выбор) (Дождь),14. Фон (Множественный выбор) (Солнце),14. Фон (Множественный выбор) (Земля),14. Фон (Множественный выбор) (Другое),14.1 Площадь штриховки (Одиночный выбор),14.2 Нажим (Одиночный выбор),Шея отделена от тела линией (Одиночный выбор)
0,Код,Возраст,Пол,1,2,3,4,5,6,7,...,249,250,251,252,1058,1059,1060,1062,1063,253
1,ЧУВ6лм1кл_6,6,мужской,NaN,NaN,Смещение влево/3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
2,ЧУВ6лм1кл_5,6,мужской,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
3,ЧУВ6лм1кл_4,6,мужской,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
4,ЧУВ5лмдс_51,5,мужской,NaN,NaN,Смещение влево/3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8578,ЧУВ5лждс (76),5,женский,Без особенностей/1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Просто линия
8579,ЧУВ5лждс (75),5,женский,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ничего из перечисленного
8580,ЧУВ6лмдс,6,мужской,NaN,NaN,NaN,Смещение вверх/4,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Просто линия
8581,1,1,женский,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,Пейзаж/5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Галстук


In [ ]:
# Очистка от неразрывных пробелов ('\xa0')

def count_xao_in_df(df):
    n = 0
    for c in df.columns:
        if isinstance(c, str):
            n += c.count("\xa0")
    for col in df.columns:
        s = df[col]
        for v in s:
            if isinstance(v, str):
                n += v.count("\xa0")
    return n

def strip_xao(x):
    if isinstance(x, str):
        return x.replace("\xa0", " ")
    return x

In [ ]:
print(f"Кол-во'\\xa0' до очистки {count_xao_in_df(df)}")

df.columns = [strip_xao(c) if isinstance(c, str) else c for c in df.columns]
df = df.map(strip_xao)

print("Кол-во '\\xa0' после очистки:", count_xao_in_df(df))

Кол-во'\xa0' до очистки 39
Кол-во '\xa0' после очистки: 0


In [ ]:
df

,Укажите код-идентификатор обследуемого (Свободный ответ),Укажите возраст обследуемого (Свободный ответ),Укажите пол обследуемого (Одиночный выбор),1 Расположение на листе (Множественный выбор) (Без особенностей/1),1 Расположение на листе (Множественный выбор) (Смещение вправо/2),1 Расположение на листе (Множественный выбор) (Смещение влево/3),1 Расположение на листе (Множественный выбор) (Смещение вверх/4),1 Расположение на листе (Множественный выбор) (Смещение вниз/5),1 Расположение на листе (Множественный выбор) (Размещение в углу/6),"1 Расположение на листе (Множественный выбор) (Изображение обрезано - есть место для дорисовки, но не дорисовано)",...,14. Фон (Множественный выбор) (Заштрихованное облако/4),14. Фон (Множественный выбор) (Пейзаж/5),14. Фон (Множественный выбор) (Пейзаж преобладает над изображением человека/6),14. Фон (Множественный выбор) (Дождь),14. Фон (Множественный выбор) (Солнце),14. Фон (Множественный выбор) (Земля),14. Фон (Множественный выбор) (Другое),14.1 Площадь штриховки (Одиночный выбор),14.2 Нажим (Одиночный выбор),Шея отделена от тела линией (Одиночный выбор)
0,Код,Возраст,Пол,1,2,3,4,5,6,7,...,249,250,251,252,1058,1059,1060,1062,1063,253
1,ЧУВ6лм1кл_6,6,мужской,NaN,NaN,Смещение влево/3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
2,ЧУВ6лм1кл_5,6,мужской,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
3,ЧУВ6лм1кл_4,6,мужской,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
4,ЧУВ5лмдс_51,5,мужской,NaN,NaN,Смещение влево/3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8578,ЧУВ5лждс (76),5,женский,Без особенностей/1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Просто линия
8579,ЧУВ5лждс (75),5,женский,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ничего из перечисленного
8580,ЧУВ6лмдс,6,мужской,NaN,NaN,NaN,Смещение вверх/4,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Просто линия
8581,1,1,женский,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,Пейзаж/5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Галстук


In [ ]:
# Создаём словарь для первой строки, чтобы обработать названия признаков

features_to_rename = dict()

for col, val in df.iloc[0].items():
    features_to_rename[col] = col

In [ ]:
def process_feature_name(val):
    if not isinstance(val, str):
        return val

    s = val

    # 1. Удаляем только неразрывные пробелы и заменяем их обычными пробелами
    s = s.replace('\xa0', ' ')

    # 2. Удаляем стандартные пометки о типе выбора
    s = s.replace('(Одиночный выбор)', '')
    s = s.replace('(Множественный выбор)', '')

    # 3. Чистим только внутренности скобок
    def parentheses_clean(subs):
        s2 = subs

        if re.search(r'/\s*\d', s2):
            s2 = re.split(r'/\s*\d', s2, maxsplit=1)[0]

        s2 = s2.replace('=>', ' ')
        s2 = re.sub(r'\d+(?:\.\d+)*', '', s2)
        s2 = re.sub(r'\s+', ' ', s2)
        return s2.strip()

    stack = []
    spans = []
    for i, ch in enumerate(s):
        if ch == '(':
            stack.append(i)
        elif ch == ')' and stack:
            start = stack.pop()
            spans.append((start, i))

    for start, end in sorted(spans, key=lambda t: t[0], reverse=True):
        txt = s[start + 1:end]

        if "=>" in txt or re.search(r'[0-9]', txt) or re.search(r'/\s*\d', txt):
            new_txt = parentheses_clean(txt)
        else:
            new_txt = re.sub(r'\s+', ' ', txt).strip()

        new_txt = new_txt.lower()
        s = s[:start + 1] + new_txt + s[end:]

    # 4. Удаляем ведущие (начальные) числа, точки и лишние разделители только в НАЧАЛЕ строки
    s = re.sub(r'^\s*(?:\d+\.)*\d*\s*', '', s)

    # 5. Удаляем лишние пробелы ПО ВСЕЙ СТРОКЕ (заменяем все последовательности пробелов на один пробел)
    s = re.sub(r' +', ' ', s)

    # 6. Нормализуем слеши: ровно один пробел до и после '/'
    s = re.sub(r'\s*/\s*', ' / ', s)
    s = re.sub(r' +', ' ', s)

    # 7. Заменяем двойные кавычки одинарными
    s = s.replace('"', "'")
    s = s.replace('«', "'")
    s = s.replace('»', "'")

    # 8. Удаляем пробелы по краям
    s = s.strip()
    return s

processed_features = {}
for k, v in features_to_rename.items():
    processed_features[k] = process_feature_name(v)

# with open("features_to_rename_processed_1.json", "w", encoding="utf-8") as f:
#     json.dump(processed_features, f, ensure_ascii=False, indent=2)

In [ ]:
# Дололнительно обрабатываем отдельные признаки

manually_renaming_features = {
    "Укажите код-идентификатор обследуемого (Свободный ответ)": "Код-идентификатор обследуемого",
    "Укажите возраст обследуемого (Свободный ответ)": "Возраст обследуемого",
    "Укажите пол обследуемого (Одиночный выбор)": "Пол обследуемого",
}

In [ ]:
for old_feat, new_feat in manually_renaming_features.items():
    processed_features[old_feat] = new_feat

# with open("features_to_rename_processed_2.json", "w", encoding="utf-8") as f:
#     json.dump(processed_features, f, ensure_ascii=False, indent=2)

In [ ]:
mismatched_keys = [c for c in df.columns if c not in processed_features]
print(f"Кол-во несовпадающих признаков: {len(mismatched_keys)}")

Кол-во несовпадающих признаков: 0


In [ ]:
# Таблица с необработанными признаками

df

,Укажите код-идентификатор обследуемого (Свободный ответ),Укажите возраст обследуемого (Свободный ответ),Укажите пол обследуемого (Одиночный выбор),1 Расположение на листе (Множественный выбор) (Без особенностей/1),1 Расположение на листе (Множественный выбор) (Смещение вправо/2),1 Расположение на листе (Множественный выбор) (Смещение влево/3),1 Расположение на листе (Множественный выбор) (Смещение вверх/4),1 Расположение на листе (Множественный выбор) (Смещение вниз/5),1 Расположение на листе (Множественный выбор) (Размещение в углу/6),"1 Расположение на листе (Множественный выбор) (Изображение обрезано - есть место для дорисовки, но не дорисовано)",...,14. Фон (Множественный выбор) (Заштрихованное облако/4),14. Фон (Множественный выбор) (Пейзаж/5),14. Фон (Множественный выбор) (Пейзаж преобладает над изображением человека/6),14. Фон (Множественный выбор) (Дождь),14. Фон (Множественный выбор) (Солнце),14. Фон (Множественный выбор) (Земля),14. Фон (Множественный выбор) (Другое),14.1 Площадь штриховки (Одиночный выбор),14.2 Нажим (Одиночный выбор),Шея отделена от тела линией (Одиночный выбор)
0,Код,Возраст,Пол,1,2,3,4,5,6,7,...,249,250,251,252,1058,1059,1060,1062,1063,253
1,ЧУВ6лм1кл_6,6,мужской,NaN,NaN,Смещение влево/3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
2,ЧУВ6лм1кл_5,6,мужской,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
3,ЧУВ6лм1кл_4,6,мужской,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
4,ЧУВ5лмдс_51,5,мужской,NaN,NaN,Смещение влево/3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8578,ЧУВ5лждс (76),5,женский,Без особенностей/1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Просто линия
8579,ЧУВ5лждс (75),5,женский,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ничего из перечисленного
8580,ЧУВ6лмдс,6,мужской,NaN,NaN,NaN,Смещение вверх/4,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Просто линия
8581,1,1,женский,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,Пейзаж/5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Галстук


In [ ]:
# Таблица с обработанными признаками

df = df.rename(columns=processed_features)
df

,Код-идентификатор обследуемого,Возраст обследуемого,Пол обследуемого,Расположение на листе (без особенностей),Расположение на листе (смещение вправо),Расположение на листе (смещение влево),Расположение на листе (смещение вверх),Расположение на листе (смещение вниз),Расположение на листе (размещение в углу),"Расположение на листе (изображение обрезано - есть место для дорисовки, но не дорисовано)",...,Фон (заштрихованное облако),Фон (пейзаж),Фон (пейзаж преобладает над изображением человека),Фон (дождь),Фон (солнце),Фон (земля),Фон (другое),Площадь штриховки,Нажим,Шея отделена от тела линией
0,Код,Возраст,Пол,1,2,3,4,5,6,7,...,249,250,251,252,1058,1059,1060,1062,1063,253
1,ЧУВ6лм1кл_6,6,мужской,NaN,NaN,Смещение влево/3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
2,ЧУВ6лм1кл_5,6,мужской,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
3,ЧУВ6лм1кл_4,6,мужской,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
4,ЧУВ5лмдс_51,5,мужской,NaN,NaN,Смещение влево/3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Шея отсутствует
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8578,ЧУВ5лждс (76),5,женский,Без особенностей/1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Просто линия
8579,ЧУВ5лждс (75),5,женский,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ничего из перечисленного
8580,ЧУВ6лмдс,6,мужской,NaN,NaN,NaN,Смещение вверх/4,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Просто линия
8581,1,1,женский,NaN,NaN,NaN,NaN,Смещение вниз/5,NaN,NaN,...,NaN,Пейзаж/5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Галстук


In [ ]:
# Вторая строка содержит номера признаков. Присвоим номера признакам 'Возраст' и 'Пол'

df.iat[0, 0] = 10000
df.iat[0, 1] = 2001
df.iat[0, 2] = 2002

In [ ]:
# Проверим типы данных в нулевой строке

row0 = df.iloc[0]
types = row0.map(lambda x: "EMPTY" if pd.isna(x) else type(x))
print(types.value_counts())

0
<class 'int'>              305
<class 'numpy.float64'>      6
Name: count, dtype: int64


In [ ]:
# import pandas as pd
# import numpy as np

# for j in range(df.shape[1]):
#     v = df.iat[0, j]

#     if pd.isna(v):
#         continue

#     col = df.columns[j]

#     if isinstance(v, (np.floating, float)):
#         xf = float(v)
#         if xf.is_integer():
#             if np.issubdtype(df.dtypes.iloc[j], np.floating):
#                 df[col] = df[col].astype("object")
#             df.iat[0, j] = int(xf)
#         continue

#     if isinstance(v, np.integer):
#         if np.issubdtype(df.dtypes.iloc[j], np.floating):
#             df[col] = df[col].astype("object")
#         df.iat[0, j] = int(v)

# row0 = df.iloc[0]

# from collections import defaultdict

# type_tag = row0.map(lambda x: "MISSING" if pd.isna(x) else type(x))
# print(f"Уникальных типов: {type_tag.nunique()}")

# by_type = defaultdict(list)
# for col, v in row0.items():
#     by_type["MISSING" if pd.isna(v) else type(v)].append((col, v))

# for t, items in by_type.items():
#     print(f"\n{t} cnt: {len(items)} == {df.shape[1]}")
#     for col, v in items[:3]:
#         print(f" - {col} => {v!r}")

In [ ]:
# Проверим уникальные типы данных по всей таблице

by_type = defaultdict(list)

for j in range(df.shape[1]):
    col_name = df.columns[j]
    for i in range(df.shape[0]):
        v = df.iat[i, j]
        t = "EMPTY" if pd.isna(v) else type(v)
        by_type[t].append((i, j, col_name, v))

print("Уникальные типы:", len(by_type))
for t, items in sorted(by_type.items(), key=lambda kv: len(kv[1]), reverse=True):
    print(f"\n{t} (cnt = {len(items)})")
    for i, j, col_name, v in items[:3]:
        print(f" - (row={i}, col_idx={j}, col={col_name!r}) => {v!r}")

KeyboardInterrupt: 

In [ ]:
row0 = df.iloc[0]
ok = row0.nunique(dropna=False) == df.shape[1]
print(f"Число уникальных значений в 0-й строке равно числу столбцов: {ok}")

Число уникальных значений в 0-й строке равно числу столбцов: True


In [ ]:
df

,Код-идентификатор обследуемого,Возраст обследуемого,Пол обследуемого,Расположение на листе (без особенностей),Расположение на листе (смещение вправо),Расположение на листе (смещение влево),Расположение на листе (смещение вверх),Расположение на листе (смещение вниз),Расположение на листе (размещение в углу),Расположение на листе (изображение обрезано),...,Дополнительно (в руке или рядом) (сумка),Дополнительно (в руке или рядом) (вариант 'ничего из перечисленного'),Фон (отсутствует),Фон (штриховка),Фон (опорная линия),Фон (заштрихованное облако),Фон (пейзаж),Фон (пейзаж преобладает над изображением человека),Фон (дождь),Шея отделена от тела линией
0,10000,2001,2002,1,2,3,4,5,6,7,...,244,245,246,247,248,249,250,251,252,253
1,2172ДД9лм3кл,9,мужской,NaN,NaN,Смещение влево/3,NaN,Смещение вниз/5,NaN,NaN,...,NaN,ничего из перечисленного,Отсутствует/1,NaN,NaN,NaN,NaN,NaN,NaN,ничего из перечисленного
2,2171ЯИ10лм3кл,10,мужской,NaN,Смещение вправо/2,Смещение влево/3,NaN,NaN,NaN,NaN,...,NaN,ничего из перечисленного,Отсутствует/1,NaN,NaN,NaN,NaN,NaN,NaN,ничего из перечисленного
3,2170МЕ9лж3кл,9,женский,Без особенностей/1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,ничего из перечисленного,Отсутствует/1,NaN,NaN,NaN,NaN,NaN,NaN,ничего из перечисленного
4,2169ГМ9лм3кл,9,мужской,Без особенностей/1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,ничего из перечисленного,Отсутствует/1,NaN,NaN,NaN,NaN,NaN,NaN,ничего из перечисленного
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4405,7,9,мужской,NaN,Смещение вправо/2,NaN,NaN,NaN,NaN,NaN,...,NaN,ничего из перечисленного,NaN,NaN,NaN,NaN,Пейзаж/5,NaN,NaN,NaN
4406,1,7,мужской,NaN,NaN,Смещение влево/3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Пейзаж/5,Пейзаж преобладает над изображением человека/6,NaN,NaN
4407,123,5,женский,NaN,NaN,Смещение влево/3,NaN,Смещение вниз/5,NaN,NaN,...,NaN,NaN,Отсутствует/1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4408,АА12,12,мужской,NaN,NaN,NaN,NaN,NaN,Размещение в углу/6,NaN,...,NaN,NaN,NaN,Штриховка/2,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
features = df.iloc[0]
print(f"Все признаки имеют уникальные названия: {features.nunique(dropna=False)} == {features.shape[0]}")

Все признаки имеют уникальные названия: 256 == 256


In [ ]:
df.to_excel("1_1_features_and_labels.xlsx", index=False)

KeyboardInterrupt: 